# Objects

In Python, everything is an object. 

In [1]:
a = 1
def hi(): ...
class Dog: ...

print(isinstance(a, object))
print(isinstance(hi, object))
print(isinstance(Dog, object))

True
True
True


Even functions and classes are objects.

**But what is an object?**

Conceptually, an object is something with an identity, a type, and a value

- Identity makes the object distinct from every other object
- Type defines the object's behaviour
- Value holds the object's data

CPython implements this as a C struct. Every object’s struct starts with the same header consisting of

- A reference count (for memory management)
- A pointer to its type

it then adds its own fields to store its value. Note that identity isn't explicitly stored anywhere - CPython uses the struct's memory address as the identity.

For example,

- `5` is an object that contains a reference count, a pointer to the `int` type, and its value, which is the number 5 in binary plus its sign and length.
- `int` is an object that contains a reference count, a pointer to the `type` type, and its value, which is a bunch of stuff that defines how int instances behave.

In [2]:
x = 5

# identity, type, value
print(id(x), type(x), x)

# the type chain: 5 → int → type → type
print(type(5), type(int), type(type))

198953870842176 <class 'int'> 5
<class 'int'> <class 'type'> <class 'type'>


# Namespaces

In Python, every "variable" is a name, which is bound to an object within a namespace.

Conceptually, a namespace is a mapping from names to objects. CPython implements them as dict attributes of objects (except for function locals), for example:

- The built-in namespace is stored in `builtins.__dict__`
- The module namespace is stored in `sys.modules[__name__].__dict__` or `globals()`
- The class namespace is stored in `Class.__dict__`
- The instance namespace is stored in `obj.__dict__`
- The function local level in internal slots for better performance

In [3]:
import builtins

x = "global x"

class Parent:
    p = "from Parent"

class Child(Parent):
    c = "from Child"
    def __init__(self):
        self.i = "from instance"

obj = Child()
obj.hi = "hi"

print("'len' in builtins.__dict__: ", "len" in builtins.__dict__)
print("'x'   in globals():         ", "x" in globals())
print("'p'   in Parent.__dict__:   ", "p" in Parent.__dict__)
print("'p'   in Child.__dict__:    ", "p" in Child.__dict__)
print("'c'   in Child.__dict__:    ", "c" in Child.__dict__)
print("'c'   in obj.__dict__:      ", "c" in obj.__dict__)
print("obj.__dict__:               ", obj.__dict__)

'len' in builtins.__dict__:  True
'x'   in globals():          True
'p'   in Parent.__dict__:    True
'p'   in Child.__dict__:     False
'c'   in Child.__dict__:     True
'c'   in obj.__dict__:       False
obj.__dict__:                {'i': 'from instance', 'hi': 'hi'}


When you use a name, CPython follows a fixed lookup order

For bare names (`x`), it uses the LEGB rule

- Local: the local namespace of the current function calls
- Enclosing: the local namespace of any outer function calls
- Global: the module namespace
- Built-in: the built-in namespace

For dotted names (`obj.x`), it searches through

- Instance namespace
- Class namespace
- Parent classes' namespaces

**But what does 'bound' actually mean?**

Let's build a mental picture compared to other types of variables:

- In C, a plain variable holds a value
- In C, a pointer holds a memory address
- In C++, a reference is like a C pointer that automatically dereferences, but it can never be re-pointed
- In Python, a name is like a C++ reference, but it can be re-pointed via the assignment operator.

# Generator vs Coroutine

When `def` or `async def` runs, it creates a function object. When you call them, however, it behaves differently dependnig on the kind of function. When you call a function defined via `async def`, it creates a coroutine object which holds a frame object so it can be paused and resumed. When you call a function defined via `def` and `yield`, it creates a generator object which can similarly be paused and resumed. Calling a plain function doesn't hand you any object.

Generators and coroutine are both functions that can be paused and resumed. The main difference comes down to intent.

Generators were designed for streaming data, that is, producing data item by item. For example....

# Concurrency

Python mainly has three ways to implement concurrency: `asyncio`, `threading`, and `multiprocessing`.

## asyncio

The `asyncio` module lets you use something similar to Javascript's event loop - it runs on a single thread and uses corporative scheduling to explicitly yield control back to the event loop. 

An important note here is that in Python, coroutines are not the units of concurrency. Awaiting a plain coroutine is synchronous. It essentially creates a nested call stack, so when you write `await coroutine()`, it immediately runs that. A coroutine only yields control back to the event loop when you await an unready Future (the event loop primitive), or by extension, Task.

Python allows you to await any awaitable object, which is anything that implements `__await__` method. Essentially it asks "give me your iterator via `__await__`". These mainly come down to

* Coroutines
* `asyncio.Future`
* `aysncio.Task` (a subclass of Future that wraps a coroutine)

Awaiting a ready Future/Task, means the coroutine immediately grabs the result and continues. Awaiting an unready Future/Task, however, pauses the coroutine and gives control to the event loop. The event loop then runs the things in its queue.

An event loop is fundamentally a callback dispatcher (a callback is a function that you pass into something, and that thing controls when it runs, compared to a standard function which you explicitly call). The event loop only knows how to run callbacks, so before putting anything in the queue, it gets wrapped in a `asyncio.Handle` object. In the case of Tasks, it points to `task._step` which executes the coroutine until its next `await`. The event loop's main execution queue is called the ready queue, which holds callbacks that are ready to run right now (first in first out).

When the awaited Future getsaits result (e.g. from an I/O event, a timer, or another task), it schedules the paused Task’s callback onto the ready queue.

Note awaiting a Task does not run it. A Task is scheduled at creation (`asyncio.create_task()` pushes it to the ready queue) and `await` just checks if it's ready - if it's finished, continue, otherwise if it's pending pause and yield to the event loop.

In [3]:
import asyncio

async def plain_worker(name):
    print(f"[{name}] Plain coroutine started (synchronous).")
    print(f"[{name}] Sleeping for 0.5s. ")
    await asyncio.sleep(0.5)
    print(f"[{name}] Plain coroutine finished!")
    return f"{name}_done"

async def worker(name, delay):
    print(f"[{name}] Started. Yielding control via 'await asyncio.sleep({delay})'...")
    await asyncio.sleep(delay)
    print(f"[{name}] Resumed and finished!")
    return f"{name}_done"

async def main():
    print("--- Main coroutine started ---")

    print("--- Scheduling task a ---")
    task_a = asyncio.create_task(worker("Worker A", 2))
    print("--- Scheduling task b ---")
    task_b = asyncio.create_task(worker("Worker B", 1))


    plain_result = await plain_worker("Plain Worker")
    print(f"Got plain coroutine result: {plain_result}\n")
    

    
    print("--- Awaiting tasks ---")
    
    result_a = await task_a
    result_b = await task_b
    
    print(f"All done! Results: {plain_result}, {result_a}, {result_b}")

# Run in Jupyter
await main()

--- Main coroutine started ---
--- Scheduling task a ---
--- Scheduling task b ---
[Plain Worker] Plain coroutine started (synchronous).
[Plain Worker] Sleeping for 0.5s. 
[Worker A] Started. Yielding control via 'await asyncio.sleep(2)'...
[Worker B] Started. Yielding control via 'await asyncio.sleep(1)'...
[Plain Worker] Plain coroutine finished!
Got plain coroutine result: Plain Worker_done

--- Awaiting tasks ---
[Worker B] Resumed and finished!
[Worker A] Resumed and finished!
All done! Results: Plain Worker_done, Worker A_done, Worker B_done


## threading


The `threading` module lets you spawn multiple operating system threads. However, because of the GIL, Python only allows one thread to be executed at a time.

## multiprocessing

The `multiprocessing` module lets you spawn separate operating system processes, each with its own memory and Python interpreter. This means you can by pass the GIL and run tasks on multiple cores, which gives true parallelism. The catch is that the processes don't share memory so communication adds complexity and there is a significantly larger overhead.

## trade-offs

Both `threading` and `asyncio` are good for I/O bound tasks. A key difference is that `asyncio` uses coorporate scheduling, where `async` tasks (coroutines) explicitly yield control back to the event loop whenever there is an `await` statement, whereas `threading` uses premptive scheduling, where the OS forcefully interupts threads and switches to another. Generally, `asyncio` is preferred because the the event loop has very small context-switching overhead and can scale a lot better. However, one disadvantage is that it cannot natively handle blaocking code like synchronous tasks or CPU-heavy tasks. Multithreading can integrate easily with legacy synchronous codebases since it gives every thread a fair slice of CPU time. However, its disadvantage is that it has slightly more overhead and can create race conditions so there is an extra complexity from managing synchronisation primitives.